# Synthetic Images Detection — Experiment Notebook

This notebook executes the experiment defined in `execution_plan.md`. It tests whether GAN-generated and diffusion-generated images share a common artifact signature that is separable from real camera-pipeline images — i.e. whether "synthetic" is a valid single detectable class.

Three feature spaces are compared:

1. **CLIP ViT-B/32 image embeddings** — semantic baseline. Expected to confirm that real and fake images of the same content are distributionally close.
2. **FFT log-magnitude spectrum** — frequency-domain pipeline traces.
3. **SRM noise residual** — spatial high-pass pipeline traces (PCA-reduced).

For each feature space we produce a UMAP scatter (color = source, marker = super-category) and a quantitative table of mean L2 distances + silhouette scores.

### Data note (deviation from the original plan)

The plan asked for "N=5 categories" and "up to 500 images per (source, category)" with explicit examples like *golden retriever, sports car, church*. The synthetic dataset `Qwerty0193/genimage-processed` only contains ≈5 images per (generator, ImageNet class), so we replaced fine-grained ImageNet classes with **5 ImageNet super-categories**:

| super-cat | # ImageNet classes | example members |
|-----------|---------------------|------------------|
| dog       | 118 | golden retriever, pug, dalmatian |
| bird      | 59  | robin, peacock, flamingo |
| vehicle   | 42  | airliner, sports car, fire engine |
| food      | 46  | pineapple, pizza, banana |
| structure | 20  | church, castle, library |

This is the smallest change that keeps both the source and the category axes of the plan intact while delivering enough samples per cell to make metrics meaningful.

## Step 1 — Environment setup

Dependencies are installed via `pip` into `.venv/` (see `execution_plan.md`, step 1):

```
pip install python-dotenv datasets transformers torch torchvision \
            umap-learn scikit-learn numpy scipy matplotlib seaborn Pillow
```

`HF_TOKEN` is loaded from `.env` (project root or experiment folder).

In [ ]:
import sys, os
from pathlib import Path

EXPERIMENT_ROOT = Path.cwd()
sys.path.insert(0, str(EXPERIMENT_ROOT))

from pipeline import (
    CACHE_DIR, RESULTS_DIR, SUPER_CATEGORIES,
    load_hf_token, imagenet_class_names, imagenet_synset_to_idx,
    extract_ai_class_idx, extract_synset, super_category_for_idx,
)

_ = load_hf_token()
print(f"HF_TOKEN ok | cache={CACHE_DIR} | results={RESULTS_DIR}")

## Step 2 — Load synthetic metadata and inspect schema

`Qwerty0193/genimage-processed` is stored as one `metadata.parquet` + one `processed_images.zip`. The `metadata.parquet` columns describe which generator each image is from (`BigGAN`, `Stable Diffusion V1.5`, ...) and which split (train/val/test). The ImageNet class is **not** in a metadata column; it is encoded in the filename:

- AI files: `<3-digit class idx>_<gen>_<sample idx>.png` → e.g. `014_biggan_00143.png` is ImageNet class 14 (`indigo bunting`).
- Real (`nature`) files: `<synset>_<sample idx>.JPEG` → e.g. `n02085620_1234.JPEG` is synset `n02085620` (`Chihuahua`).

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

meta_path = hf_hub_download(
    repo_id="Qwerty0193/genimage-processed",
    filename="metadata.parquet",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"],
)
meta = pd.read_parquet(meta_path)

print(f"rows: {len(meta)}, columns: {list(meta.columns)}")
meta[["generator", "label_name"]].value_counts().to_frame("n")

## Step 3 — Select 5 super-categories

`SUPER_CATEGORIES` (defined in `pipeline.py`) maps each super-category to a curated set of ImageNet-1k class indices. For each (BigGAN-ai, SDv1.5-ai) row we extract the class index from the filename and assign a super-category. For each (real) row we extract the synset and look up the class index from a standard ImageNet-1k mapping.

In [ ]:
names = imagenet_class_names()
syn_to_idx = imagenet_synset_to_idx()
for cat, ids in SUPER_CATEGORIES.items():
    examples = ", ".join(names[i] for i in sorted(ids)[:5])
    print(f"{cat:10s} ({len(ids):3d} classes): e.g. {examples}, ...")

meta["ai_class_idx"] = meta["file_path"].map(extract_ai_class_idx)
meta["synset"] = meta["file_path"].map(extract_synset)
meta["nature_class_idx"] = meta["synset"].map(lambda s: syn_to_idx.get(s) if s else None)
meta["class_idx"] = meta["ai_class_idx"].fillna(meta["nature_class_idx"])
meta["super_cat"] = meta["class_idx"].map(
    lambda x: super_category_for_idx(int(x)) if pd.notna(x) else None
)

(meta.dropna(subset=["super_cat"])
     .groupby(["generator", "label_name", "super_cat"]).size()
     .unstack(fill_value=0))

## Steps 4 + 5 — Real dataset and sampling

Real images come from the `ILSVRC/imagenet-1k` *validation* split (50 images per ImageNet class). We stream the split, keep only rows whose label is in any of the 5 super-categories, and save up to 500 images per super-cat into `cache/real/<super_cat>/`.

Synthetic images come from the relevant rows of `Qwerty0193/genimage-processed`, extracted from `processed_images.zip` into `cache/synthetic/<source>/<super_cat>/`. We use **all** available BigGAN-ai and SDv1.5-ai images for each super-cat (well below the 500 cap).

These two steps are performed by the helper scripts in `scripts/08_prepare_synthetic.py` and `scripts/10_prepare_real.py`. We just verify the manifests here.

In [ ]:
synth_manifest = pd.read_csv(CACHE_DIR / "synthetic_manifest.csv")
real_manifest = pd.read_csv(CACHE_DIR / "real_manifest.csv")
all_manifest = pd.concat([real_manifest, synth_manifest], ignore_index=True)
print(f"total images: {len(all_manifest)}")
all_manifest.groupby(["source", "super_cat"]).size().unstack(fill_value=0)

## Step 6 — Preprocess images

All images are stored on disk as 224×224 JPEG. We load them as a `(N, 224, 224, 3)` uint8 array, which we feed into all three feature extractors (FFT/SRM use the array directly, CLIP normalises internally via `CLIPProcessor`).

In [ ]:
import numpy as np

feats_npz = CACHE_DIR / "features.npz"
if not feats_npz.is_file():
    raise SystemExit(
        "Run scripts/12_extract_features.py first to build features.npz "
        "(takes ~5 min on CPU). The notebook reads the cached features."
    )
feats = np.load(feats_npz, allow_pickle=True)
sources = np.array(feats["source"])
super_cats = np.array(feats["super_cat"])
class_idx = np.array(feats["class_idx"]).astype(int)
clip_feats = feats["clip"].astype(np.float32)
fft_feats = feats["fft"].astype(np.float32)
srm_feats = feats["srm"].astype(np.float32)
print(
    f"clip {clip_feats.shape} | fft {fft_feats.shape} | srm {srm_feats.shape}"
)

## Step 7 — Baseline 1: Semantic (CLIP) + UMAP

**Expected result.** If real and fake images of the same content are distributionally close (as the writeup argues: generators are explicitly trained to minimise this distance), CLIP — a semantic encoder — should cluster them together by content, regardless of source.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

display(Image(filename=str(RESULTS_DIR / "umap_clip.png")))

**Observation.** Four content clusters are visible (dog top-left, food top-right, vehicle/structure/bird bottom, etc.). Within each cluster, the three sources (blue = real, red = BigGAN, green = SDv1.5) are heavily intermixed. CLIP cannot separate sources — semantic content dominates. ✅ matches the writeup's prediction that distributional separability collapses by construction.

## Step 8 — Experiment 2a: FFT magnitude + UMAP

**Expected if the single-class hypothesis holds.** BigGAN and SDv1.5 should cluster together (small inter-source distance) and both should be far from real (large real-vs-fake distance), while different content within real should remain close (small same-pipeline reference).

In [ ]:
display(Image(filename=str(RESULTS_DIR / "umap_fft.png")))

**Observation.** The frequency space exhibits **two** clear clusters: BigGAN (red) forms its own isolated cluster, while real (blue) and SDv1.5 (green) are mixed in a single elongated cluster. SDv1.5 is *closer to real than to BigGAN* in FFT space. This is direct visual evidence **against** the single-class hypothesis — GAN and diffusion produce qualitatively different frequency signatures.

## Step 9 — Experiment 2b: SRM residual + UMAP

We apply a fixed 3×3 SRM high-pass kernel per channel, flatten the residual, then PCA-reduce to 256 dimensions (per the plan). UMAP runs on the PCA-reduced features.

In [ ]:
display(Image(filename=str(RESULTS_DIR / "umap_srm.png")))

**Observation.** A single diffuse blob with no clear separation by either source or super-category. The naive SRM residual (used as a flat feature vector, then PCA'd) primarily captures content-level high-frequency edges rather than pipeline noise statistics. The classical use of SRM for forensic detection relies on *statistical aggregation* of the residual (e.g. co-occurrence matrices), not the residual itself — the negative result here is consistent with that. See the writeup's `Final Comments`.

## Steps 10 + 11 — Metrics table

Five measurements per feature space:

1. **intra (same source & cat)** — mean off-diagonal L2 distance within each (source, super-cat) cell, averaged.
2. **inter biggan-vs-sdv15 (same cat)** — mean L2 distance between BigGAN and SDv1.5 of the same super-cat.
3. **inter real-vs-fake (same cat)** — mean L2 distance between real and (BigGAN ∪ SDv1.5) of the same super-cat.
4. **inter real-cat A vs real-cat B** — mean L2 distance between two *different* super-cats inside `real` (the same-pipeline reference).
5. **silhouette 3-class (real/GAN/diff)** — sklearn silhouette over the 3-way label, ignoring category. Additional silhouettes are reported for `real vs fake` and `GAN vs diffusion` contrasts.

In [ ]:
metrics_pivot = pd.read_csv(RESULTS_DIR / "metrics_pivot.csv", index_col=0)
metrics_pivot[["clip", "fft", "srm"]]

## Interpretation

| Comparison | CLIP | FFT | SRM | Reading |
|---|---|---|---|---|
| intra (same source & cat) | 0.84 | 57.6 | 3.36 | baseline tightness |
| BigGAN vs SDv1.5 (same cat) | 0.86 | **78.8** | 2.86 | **FFT separates the two fake families more than anything else** |
| real vs fake (same cat) | 0.88 | 67.9 | 3.95 | similar magnitude to intra in CLIP → no separation; intermediate in FFT |
| real cat A vs real cat B | **1.03** | 61.2 | **5.10** | content matters most in CLIP and SRM |
| silhouette real vs fake | 0.01 | 0.06 | -0.06 | weak separability everywhere |
| silhouette GAN vs diffusion (fakes only) | 0.04 | **0.25** | 0.17 | **FFT cleanly tells BigGAN from SDv1.5** |

**Headline findings.**

- *CLIP* confirms the writeup's premise: real and synthetic images of the same content are essentially indistinguishable in semantic space.
- *FFT* refutes the "single fake class" hypothesis. BigGAN sits in a frequency neighbourhood of its own; SDv1.5 lives much closer to real photos than to BigGAN. The same-pipeline reference (real-cat-A vs real-cat-B, 61.2) is **smaller** than the BigGAN–SDv1.5 distance (78.8), meaning the two synthetic mechanisms are further apart from each other than two random content classes inside the real pipeline.
- *SRM*, used as a flat residual, is dominated by content and produces no clean source separation — a known limitation of this simple feature design.

Concretely: detection that treats "AI-generated" as a single class will undersell what is actually visible in frequency space (where GAN and diffusion form *different* artifact families), and oversell separability where it does not exist (SRM here).

See `writeup.md` for fuller context and limitations.